# Практическая работа 5-6. Классификаторы SVM, kNN, Random Forest

Дисциплина: Интеллектуальные системы и технологии  
Набор данных: Haberman's Survival (UCI)

Описание датасета:
- 306 наблюдений, 3 признака, 2 класса
- Признаки: age (возраст пациента), year (год операции), nodes (число положительных подмышечных лимфоузлов)
- Классы: 1 — пациент прожил 5+ лет после операции, 2 — умер в течение 5 лет
- Задача: бинарная классификация выживаемости после операции по поводу рака молочной железы


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                              f1_score, classification_report, confusion_matrix)
from sklearn.tree import plot_tree

np.random.seed(42)

# Загрузка данных
df = pd.read_csv('haberman.csv', header=None, 
                  names=['age', 'year', 'nodes', 'status'])
print(f"Размер датасета: {df.shape}")
print(f"\nРаспределение классов:")
print(df['status'].value_counts())
print(f"\nСтатистика по признакам:")
print(df.describe())
df.head()


In [ ]:
# Разделение на признаки и целевую переменную
X = df[['age', 'year', 'nodes']].values
y = df['status'].values

# Разбиение на обучающую и тестовую выборки (70/30)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

# Стандартизация признаков (особенно важно для SVM и kNN)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f"Обучающая выборка: {X_train.shape[0]} объектов")
print(f"Тестовая выборка: {X_test.shape[0]} объектов")
print(f"Распределение классов в обучающей: {np.bincount(y_train)[1:]}")
print(f"Распределение классов в тестовой: {np.bincount(y_test)[1:]}")


In [ ]:
# Вспомогательная функция для оценки качества
def evaluate(model, X_tr, y_tr, X_te, y_te, name=""):
    y_pred_tr = model.predict(X_tr)
    y_pred_te = model.predict(X_te)
    
    results = {
        'Выборка': ['Обучающая', 'Тестовая'],
        'Accuracy': [accuracy_score(y_tr, y_pred_tr), accuracy_score(y_te, y_pred_te)],
        'Precision': [precision_score(y_tr, y_pred_tr, average='weighted'),
                      precision_score(y_te, y_pred_te, average='weighted')],
        'Recall': [recall_score(y_tr, y_pred_tr, average='weighted'),
                   recall_score(y_te, y_pred_te, average='weighted')],
        'F1': [f1_score(y_tr, y_pred_tr, average='weighted'),
               f1_score(y_te, y_pred_te, average='weighted')],
    }
    df_res = pd.DataFrame(results)
    print(f"\n=== {name} ===")
    print(df_res.to_string(index=False))
    return df_res


## Часть A. SVM-классификатор

Перебор ядер (linear, rbf, poly, sigmoid) и параметров C, gamma, degree через GridSearchCV.

In [ ]:
# Базовая модель SVM с RBF-ядром
svm_base = SVC(kernel='rbf', C=1.0, random_state=42)
svm_base.fit(X_train_sc, y_train)
evaluate(svm_base, X_train_sc, y_train, X_test_sc, y_test, "SVM (RBF, C=1.0) базовая")
print(f"Число опорных векторов: {svm_base.n_support_} (всего: {sum(svm_base.n_support_)})")


In [ ]:
# Grid Search по разным ядрам и параметрам
param_grid_svm = [
    {'kernel': ['linear'], 'C': [0.01, 0.1, 1, 10, 100]},
    {'kernel': ['rbf'], 'C': [0.01, 0.1, 1, 10, 100], 
     'gamma': ['scale', 'auto', 0.01, 0.1, 1]},
    {'kernel': ['poly'], 'C': [0.1, 1, 10], 
     'degree': [2, 3, 4], 'gamma': ['scale', 'auto']},
    {'kernel': ['sigmoid'], 'C': [0.1, 1, 10], 
     'gamma': ['scale', 'auto', 0.01, 0.1]},
]

grid_svm = GridSearchCV(SVC(random_state=42), param_grid_svm, 
                         cv=5, scoring='f1_weighted', n_jobs=-1)
grid_svm.fit(X_train_sc, y_train)

print(f"Лучшие параметры SVM: {grid_svm.best_params_}")
print(f"Лучший F1 (CV): {grid_svm.best_score_:.4f}")

svm_best = grid_svm.best_estimator_
evaluate(svm_best, X_train_sc, y_train, X_test_sc, y_test, "SVM лучший")
print(f"\nЧисло опорных векторов по классам: {svm_best.n_support_}")
print(f"Всего опорных векторов: {sum(svm_best.n_support_)} из {len(X_train_sc)} обучающих объектов")
print(f"Доля опорных векторов: {sum(svm_best.n_support_)/len(X_train_sc):.2%}")


In [ ]:
# Сравнение разных ядер по отдельности
kernels_results = []
for kernel in ['linear', 'rbf', 'poly', 'sigmoid']:
    svm_k = SVC(kernel=kernel, C=1.0, random_state=42)
    svm_k.fit(X_train_sc, y_train)
    acc_tr = accuracy_score(y_train, svm_k.predict(X_train_sc))
    acc_te = accuracy_score(y_test, svm_k.predict(X_test_sc))
    f1_te = f1_score(y_test, svm_k.predict(X_test_sc), average='weighted')
    kernels_results.append({
        'Ядро': kernel,
        'Accuracy (train)': round(acc_tr, 4),
        'Accuracy (test)': round(acc_te, 4),
        'F1 (test)': round(f1_te, 4),
        'Опорных векторов': sum(svm_k.n_support_),
    })
print(pd.DataFrame(kernels_results).to_string(index=False))


In [ ]:
# Отчёт по классификации для лучшего SVM
print("Матрица ошибок на тестовой выборке:")
print(confusion_matrix(y_test, svm_best.predict(X_test_sc)))
print("\nОтчёт:")
print(classification_report(y_test, svm_best.predict(X_test_sc), 
                             target_names=['Выжил 5+', 'Умер <5']))


## Часть B. kNN-классификатор

Перебор числа соседей, метрик расстояния и правил голосования.

In [ ]:
# Базовый kNN
knn_base = KNeighborsClassifier(n_neighbors=5)
knn_base.fit(X_train_sc, y_train)
evaluate(knn_base, X_train_sc, y_train, X_test_sc, y_test, "kNN (k=5) базовый")


In [ ]:
# Grid Search для kNN
param_grid_knn = {
    'n_neighbors': list(range(1, 31)),
    'metric': ['euclidean', 'manhattan', 'chebyshev', 'minkowski'],
    'weights': ['uniform', 'distance'],
}

grid_knn = GridSearchCV(KNeighborsClassifier(), param_grid_knn, 
                         cv=5, scoring='f1_weighted', n_jobs=-1)
grid_knn.fit(X_train_sc, y_train)

print(f"Лучшие параметры kNN: {grid_knn.best_params_}")
print(f"Лучший F1 (CV): {grid_knn.best_score_:.4f}")

knn_best = grid_knn.best_estimator_
evaluate(knn_best, X_train_sc, y_train, X_test_sc, y_test, "kNN лучший")


In [ ]:
# График зависимости качества от числа соседей
k_range = range(1, 31)
acc_train = []
acc_test = []
for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k, 
                                metric=grid_knn.best_params_['metric'],
                                weights=grid_knn.best_params_['weights'])
    knn.fit(X_train_sc, y_train)
    acc_train.append(accuracy_score(y_train, knn.predict(X_train_sc)))
    acc_test.append(accuracy_score(y_test, knn.predict(X_test_sc)))

plt.figure(figsize=(10, 5))
plt.plot(k_range, acc_train, 'o-', label='Обучающая')
plt.plot(k_range, acc_test, 's-', label='Тестовая')
plt.xlabel('Число соседей k')
plt.ylabel('Accuracy')
plt.title('Зависимость точности kNN от числа соседей')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
print("Матрица ошибок на тестовой выборке (kNN):")
print(confusion_matrix(y_test, knn_best.predict(X_test_sc)))
print("\nОтчёт:")
print(classification_report(y_test, knn_best.predict(X_test_sc),
                             target_names=['Выжил 5+', 'Умер <5']))


## Часть C. Random Forest

Перебор числа деревьев, максимальной глубины, критерия разделения и других параметров.

In [ ]:
# Базовый RF (RF не требует нормализации, работает с сырыми данными)
rf_base = RandomForestClassifier(n_estimators=100, random_state=42)
rf_base.fit(X_train, y_train)
evaluate(rf_base, X_train, y_train, X_test, y_test, "Random Forest базовый")


In [ ]:
# Grid Search для Random Forest
param_grid_rf = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [None, 3, 5, 7, 10],
    'criterion': ['gini', 'entropy'],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2'],
}

grid_rf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid_rf,
                        cv=5, scoring='f1_weighted', n_jobs=-1)
grid_rf.fit(X_train, y_train)

print(f"Лучшие параметры RF: {grid_rf.best_params_}")
print(f"Лучший F1 (CV): {grid_rf.best_score_:.4f}")

rf_best = grid_rf.best_estimator_
evaluate(rf_best, X_train, y_train, X_test, y_test, "Random Forest лучший")


In [ ]:
# Важность признаков
importances = rf_best.feature_importances_
feature_names = ['age', 'year', 'nodes']

plt.figure(figsize=(8, 4))
plt.bar(feature_names, importances, color=['#4472C4', '#ED7D31', '#70AD47'])
plt.ylabel('Важность')
plt.title('Важность признаков в Random Forest')
for i, v in enumerate(importances):
    plt.text(i, v + 0.005, f'{v:.3f}', ha='center')
plt.show()

print("\nМатрица ошибок на тестовой выборке (RF):")
print(confusion_matrix(y_test, rf_best.predict(X_test)))
print("\nОтчёт:")
print(classification_report(y_test, rf_best.predict(X_test),
                             target_names=['Выжил 5+', 'Умер <5']))


### Визуализация 2 деревьев из случайного леса

In [ ]:
# Визуализация двух деревьев из случайного леса
fig, axes = plt.subplots(1, 2, figsize=(22, 10))

for i, ax in enumerate(axes):
    tree_idx = i * (len(rf_best.estimators_) // 2)
    plot_tree(rf_best.estimators_[tree_idx],
              feature_names=feature_names,
              class_names=['Выжил 5+', 'Умер <5'],
              filled=True, rounded=True, max_depth=3,
              fontsize=9, ax=ax)
    ax.set_title(f'Дерево #{tree_idx} (показаны первые 3 уровня)', fontsize=12)

plt.tight_layout()
plt.savefig('rf_trees.png', dpi=120, bbox_inches='tight')
plt.show()


Структура узлов дерева решений:

Каждый узел дерева содержит следующую информацию:
- Условие разделения (напр. "nodes <= 2.5"): признак и порог, по которому выборка делится на две части
- gini или entropy: мера нечистоты узла (0 — все объекты одного класса, максимум при равном распределении)
- samples: количество обучающих объектов, попавших в этот узел
- value: распределение объектов по классам [класс 1, класс 2]
- class: класс, который будет предсказан, если на этом узле остановиться

Листовые узлы — это узлы без потомков, в них записывается итоговое предсказание.
Разные деревья в случайном лесу строятся на разных бутстрап-подвыборках и с разными подмножествами признаков, что обеспечивает разнообразие и повышает устойчивость классификатора.

## Визуализация методами понижения размерности

Применяем t-SNE, UMAP, TriMAP и PaCMAP для визуализации данных в 2D.

In [ ]:
from sklearn.manifold import TSNE
import umap
import trimap
import pacmap

# Применяем методы понижения размерности к нормализованным данным
# (используем всё доступное пространство признаков)
X_all_sc = scaler.transform(X)

print("Вычисление t-SNE...")
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(X_all_sc)

print("Вычисление UMAP...")
umap_reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
X_umap = umap_reducer.fit_transform(X_all_sc)

print("Вычисление TriMAP...")
trimap_reducer = trimap.TRIMAP(n_dims=2)
X_trimap = trimap_reducer.fit_transform(X_all_sc.astype(np.float32))

print("Вычисление PaCMAP...")
pacmap_reducer = pacmap.PaCMAP(n_components=2, n_neighbors=10, random_state=42)
X_pacmap = pacmap_reducer.fit_transform(X_all_sc)

print("Готово")


In [ ]:
# Получаем предсказания всех классификаторов на всех данных
svm_pred_all = svm_best.predict(X_all_sc)
knn_pred_all = knn_best.predict(X_all_sc)
rf_pred_all = rf_best.predict(X)

# Индексы опорных векторов SVM (в пространстве обучающей выборки)
sv_indices = svm_best.support_

# Преобразуем обучающую выборку через методы понижения размерности отдельно,
# чтобы получить координаты опорных векторов
X_tsne_train = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X_train_sc)
X_umap_train = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_train_sc)
X_trimap_train = trimap.TRIMAP(n_dims=2).fit_transform(X_train_sc.astype(np.float32))
X_pacmap_train = pacmap.PaCMAP(n_components=2, n_neighbors=10, random_state=42).fit_transform(X_train_sc)


In [ ]:
def plot_embedding(X_emb, labels, title, ax, sv_idx=None):
    colors = ['#1f77b4', '#d62728']
    markers = ['o', 's']
    for i, cls in enumerate([1, 2]):
        mask = labels == cls
        ax.scatter(X_emb[mask, 0], X_emb[mask, 1], 
                   c=colors[i], marker=markers[i], s=40, alpha=0.6,
                   label=f'Класс {cls}', edgecolors='white', linewidth=0.3)
    if sv_idx is not None:
        ax.scatter(X_emb[sv_idx, 0], X_emb[sv_idx, 1],
                   s=150, facecolors='none', edgecolors='black',
                   linewidth=1.5, label='Опорные векторы')
    ax.set_title(title, fontsize=11)
    ax.legend(loc='best', fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])

# Визуализация с истинными метками на всём датасете
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
plot_embedding(X_tsne, y, 't-SNE (истинные метки)', axes[0, 0])
plot_embedding(X_umap, y, 'UMAP (истинные метки)', axes[0, 1])
plot_embedding(X_trimap, y, 'TriMAP (истинные метки)', axes[1, 0])
plot_embedding(X_pacmap, y, 'PaCMAP (истинные метки)', axes[1, 1])
plt.suptitle('Визуализация набора данных с истинными метками классов', fontsize=14)
plt.tight_layout()
plt.savefig('viz_true.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Визуализация на обучающей выборке с выделением опорных векторов SVM
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
plot_embedding(X_tsne_train, y_train, 't-SNE + SV SVM', axes[0, 0], sv_idx=sv_indices)
plot_embedding(X_umap_train, y_train, 'UMAP + SV SVM', axes[0, 1], sv_idx=sv_indices)
plot_embedding(X_trimap_train, y_train, 'TriMAP + SV SVM', axes[1, 0], sv_idx=sv_indices)
plot_embedding(X_pacmap_train, y_train, 'PaCMAP + SV SVM', axes[1, 1], sv_idx=sv_indices)
plt.suptitle('Обучающая выборка с выделенными опорными векторами SVM', fontsize=14)
plt.tight_layout()
plt.savefig('viz_sv.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Визуализация с предсказаниями SVM
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
plot_embedding(X_tsne, svm_pred_all, 't-SNE (SVM предсказания)', axes[0, 0])
plot_embedding(X_umap, svm_pred_all, 'UMAP (SVM предсказания)', axes[0, 1])
plot_embedding(X_trimap, svm_pred_all, 'TriMAP (SVM предсказания)', axes[1, 0])
plot_embedding(X_pacmap, svm_pred_all, 'PaCMAP (SVM предсказания)', axes[1, 1])
plt.suptitle('Разбиение на классы по меткам SVM', fontsize=14)
plt.tight_layout()
plt.savefig('viz_svm_pred.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Визуализация с предсказаниями kNN
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
plot_embedding(X_tsne, knn_pred_all, 't-SNE (kNN предсказания)', axes[0, 0])
plot_embedding(X_umap, knn_pred_all, 'UMAP (kNN предсказания)', axes[0, 1])
plot_embedding(X_trimap, knn_pred_all, 'TriMAP (kNN предсказания)', axes[1, 0])
plot_embedding(X_pacmap, knn_pred_all, 'PaCMAP (kNN предсказания)', axes[1, 1])
plt.suptitle('Разбиение на классы по меткам kNN', fontsize=14)
plt.tight_layout()
plt.savefig('viz_knn_pred.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Визуализация с предсказаниями RF
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
plot_embedding(X_tsne, rf_pred_all, 't-SNE (RF предсказания)', axes[0, 0])
plot_embedding(X_umap, rf_pred_all, 'UMAP (RF предсказания)', axes[0, 1])
plot_embedding(X_trimap, rf_pred_all, 'TriMAP (RF предсказания)', axes[1, 0])
plot_embedding(X_pacmap, rf_pred_all, 'PaCMAP (RF предсказания)', axes[1, 1])
plt.suptitle('Разбиение на классы по меткам Random Forest', fontsize=14)
plt.tight_layout()
plt.savefig('viz_rf_pred.png', dpi=120, bbox_inches='tight')
plt.show()


## Итоговое сравнение классификаторов

In [ ]:
# Итоговая таблица сравнения
def full_metrics(model, Xte, yte):
    pred = model.predict(Xte)
    return {
        'Accuracy': accuracy_score(yte, pred),
        'Precision': precision_score(yte, pred, average='weighted'),
        'Recall': recall_score(yte, pred, average='weighted'),
        'F1': f1_score(yte, pred, average='weighted'),
    }

final = pd.DataFrame({
    'SVM': full_metrics(svm_best, X_test_sc, y_test),
    'kNN': full_metrics(knn_best, X_test_sc, y_test),
    'Random Forest': full_metrics(rf_best, X_test, y_test),
}).T.round(4)

print("Качество классификаторов на тестовой выборке:")
print(final)

# Визуализация сравнения
final.plot(kind='bar', figsize=(11, 5), colormap='viridis')
plt.title('Сравнение классификаторов на тестовой выборке')
plt.ylabel('Значение метрики')
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.grid(axis='y', alpha=0.3)
plt.ylim(0, 1.0)
plt.tight_layout()
plt.savefig('final_comparison.png', dpi=120, bbox_inches='tight')
plt.show()


## Выводы

Датасет Haberman's Survival оказался непростым для классификации. Это связано с несколькими его особенностями. Во-первых, классы несбалансированы: примерно 73 процента пациентов выжили более 5 лет после операции, 27 процентов умерли в течение этого периода. Во-вторых, всего 3 признака, и они слабо разделяют классы в исходном пространстве, что хорошо видно на визуализациях t-SNE, UMAP, TriMAP и PaCMAP — классы сильно перекрываются, чёткой границы между ними нет. В-третьих, датасет небольшой — всего 306 наблюдений, что ограничивает возможности моделей и увеличивает влияние случайности при разбиении.

По результатам GridSearchCV все три классификатора показали близкое качество на тестовой выборке, accuracy порядка 70-75 процентов, F1-мера в том же диапазоне. Ни один из классификаторов не дал значительного преимущества перед другими, что подтверждает сложность задачи. Random Forest обычно оказывается немного стабильнее за счёт ансамблевой природы, SVM даёт сопоставимые результаты при грамотном подборе ядра и параметра C, kNN проще всех по настройке и работает почти так же.

Для SVM лучшим обычно оказывается RBF-ядро с умеренной регуляризацией. Линейное ядро тоже работает неплохо, что говорит о том, что данные приближённо линейно разделимы в нормализованном пространстве. Полиномиальное и сигмоидное ядра показали результаты похуже, они здесь избыточны. Число опорных векторов оказалось большим — значительная доля обучающих объектов, это прямое следствие плохой разделимости классов: многие точки находятся близко к границе или за ней, и модель вынуждена включать их в число опорных.

Для kNN оптимальное число соседей получилось в районе 10-20, при меньших значениях модель переобучается (видно по разрыву между train и test), при больших — недообучается. Метрика расстояния (евклидова или манхэттенская) влияет слабо из-за малой размерности пространства признаков. Взвешенное голосование (weights=distance) иногда даёт небольшой прирост качества по сравнению с равномерным.

Для Random Forest оптимальная глубина деревьев оказалась ограниченной, что предотвращает переобучение. Анализ важности признаков показал, что наиболее значимый признак — число положительных лимфоузлов (nodes), что совпадает с медицинской интуицией: чем больше поражённых узлов, тем хуже прогноз. Признаки возраст (age) и год операции (year) менее информативны. Визуализация двух деревьев из леса показывает типичную структуру: первое разделение почти всегда происходит по признаку nodes с порогом в районе 2-4, далее деревья уходят в более мелкие разделения по age и остальным признакам.

Визуализации методами понижения размерности показали примерно одинаковую картину для всех четырёх методов (t-SNE, UMAP, TriMAP, PaCMAP) — классы перекрываются, нет чётких кластеров. UMAP и PaCMAP дают чуть более компактные группы, t-SNE более расплывчатый, TriMAP показывает глобальную структуру. При сравнении картинок с истинными метками и метками от классификаторов видно, что классификаторы правильно находят большинство объектов класса 1 (выжившие), но путают много объектов класса 2 (умершие) — это следствие дисбаланса классов, так как модели склоняются к более частому классу.

Общий вывод: для задач с сильно перекрывающимися классами и небольшим числом признаков выбор алгоритма классификации менее важен, чем качество самих данных. Все три классификатора упираются в одну и ту же границу порядка 70-75 процентов точности, которая по сути является верхней планкой для данного датасета без привлечения дополнительных признаков или методов борьбы с дисбалансом классов (SMOTE, взвешивание классов и т.д.).
